# Лабораторна робота 2. Згорткові нейронні мережі для задач класифікації

**Варіант 3** — датасет [AID (Aerial Image Dataset)](https://www.kaggle.com/datasets/jiayuanchengala/aid-scene-classification-datasets): аерофотознімки, 30 класів сцен (аеропорт, пляж, міст, житлова забудова тощо).

**Виконавці:** _(вказати прізвища)_

Ноутбук розрахований на запуск у **Google Colab з GPU** (`Runtime → Change runtime type → T4 GPU`).

| Завдання | Статус |
|---|---|
| 1. Завантаження датасету | ✅ |
| 2. Аналіз даних, поділ на train/val/test | ✅ |
| 3. ResNet18, заморожена основа, новий класифікатор | ✅ |
| 4. Навчання з аугментацією | 📝 заготовка |
| 5. Fine-tuning останнього згорткового блоку | ✅ |
| 6. Компактна CNN-учень | 📝 заготовка |
| 7. Дистиляція знань | 📝 заготовка |
| 8–10. Аналіз помилок, Grad-CAM, підсумки | 📝 заготовка |

## 0. Підготовка середовища

In [ ]:
# Перевіряємо, що Colab видав GPU (має бути видно, наприклад, Tesla T4)
!nvidia-smi

In [ ]:
# kagglehub зазвичай уже встановлений у Colab, але про всяк випадок
!pip -q install kagglehub

In [ ]:
import os, json, copy, time, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.transforms import v2

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# ---------------- Загальні налаштування ----------------
SEED = 42            # один seed для всіх експериментів -> однаковий поділ даних і відтворюваність
IMG_SIZE = 224       # стандартний розмір входу для моделей, навчених на ImageNet
BATCH_SIZE = 64
NUM_WORKERS = 2      # у Colab 2 CPU-ядра
# Воркери DataLoader запускаємо через fork: так вони бачать кеш зображень у пам'яті без копіювання
# (у Python 3.14+ за замовчуванням інший метод, з яким класи, оголошені в ноутбуці, не працюють)
MP_CONTEXT = 'fork' if NUM_WORKERS > 0 else None

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'   # mixed precision (fp16) пришвидшує навчання на GPU приблизно вдвічі
print('device:', device)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
torch.backends.cudnn.benchmark = True

In [ ]:
# Куди зберігати ваги моделей, поділ даних та результати.
# Сесія Colab може перезапуститися, тому краще увімкнути Google Drive (USE_DRIVE = True):
# тоді в наступних завданнях (наприклад, для дистиляції) можна просто завантажити готового вчителя.
USE_DRIVE = False

CKPT_DIR = Path('/content/checkpoints') if os.path.exists('/content') else Path('checkpoints')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = Path('/content/drive/MyDrive/lab2_aid')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print('checkpoints ->', CKPT_DIR)

## 1. Завантаження датасету

Використовуємо `kagglehub` — він завантажує датасет у кеш (`/root/.cache/kagglehub/...`).
Якщо Kaggle попросить авторизацію: додайте у Colab Secrets (іконка 🔑 зліва) `KAGGLE_USERNAME` і `KAGGLE_KEY`
або виконайте `kagglehub.login()`.

Датасет може містити кілька папок, тому нижче автоматично шукаємо директорію, в якій лежать підпапки-класи із зображеннями.

In [ ]:
import kagglehub

dataset_path = Path(kagglehub.dataset_download('jiayuanchengala/aid-scene-classification-datasets'))
print('Завантажено в:', dataset_path)
for p in sorted(dataset_path.iterdir()):
    print('  ', p.name)

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}


def find_class_root(root):
    # Шукаємо папку, всередині якої найбільше підпапок із зображеннями (це і є папки-класи).
    # Папки з train/test/val у шляху пропускаємо — поділ робимо самі.
    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        n_classes = 0
        for d in dirnames:
            files = os.listdir(os.path.join(dirpath, d))
            if any(Path(f).suffix.lower() in IMG_EXT for f in files):
                n_classes += 1
        if n_classes >= 2:
            candidates.append((Path(dirpath), n_classes))
    for c in candidates:
        print('кандидат:', c)
    split_words = {'train', 'test', 'val', 'valid', 'validation'}
    best = max(candidates, key=lambda c: (c[1], not (set(p.lower() for p in c[0].parts) & split_words)))
    return best[0]


DATA_DIR = find_class_root(dataset_path)   # за потреби можна вказати шлях вручну
print('\nПапка з класами:', DATA_DIR)

## 2. Аналіз даних і поділ на вибірки

In [ ]:
classes = sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir())
class_to_idx = {c: i for i, c in enumerate(classes)}
NUM_CLASSES = len(classes)

records = []
for c in classes:
    for f in sorted((DATA_DIR / c).iterdir()):
        if f.suffix.lower() in IMG_EXT:
            records.append((str(f.relative_to(DATA_DIR)), c, class_to_idx[c]))
df = pd.DataFrame(records, columns=['path', 'class', 'label'])

print(f'Кількість класів: {NUM_CLASSES}')
print(f'Кількість зображень: {len(df)}')
with Image.open(DATA_DIR / df.path[0]) as im:
    print('Розмір зображення (приклад):', im.size, im.mode)

In [ ]:
# Розподіл зображень за класами
counts = df['class'].value_counts().sort_values()

plt.figure(figsize=(8, 9))
plt.barh(counts.index, counts.values, color='steelblue')
plt.xlabel('Кількість зображень')
plt.title('Розподіл зображень за класами (AID)')
plt.tight_layout()
plt.show()

print(f'min = {counts.min()} ({counts.idxmin()}), max = {counts.max()} ({counts.idxmax()}), '
      f'співвідношення max/min = {counts.max() / counts.min():.2f}')

**Висновок щодо розподілу.** Класи в AID помірно незбалансовані (приблизно від 220 до 420 зображень на клас).
Через це, крім accuracy, будемо рахувати **macro-F1** — вона однаково враховує всі класи, незалежно від їх розміру.
Поділ на вибірки робимо **стратифікованим**, щоб пропорції класів у train/val/test були однаковими.

In [ ]:
# Приклади: по одному зображенню з кожного класу
n_cols = 6
n_rows = int(np.ceil(NUM_CLASSES / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.6 * n_cols, 2.8 * n_rows))
for ax in axes.flat:
    ax.axis('off')
for ax, c in zip(axes.flat, classes):
    path = df[df['class'] == c].sample(1, random_state=SEED).path.iloc[0]
    ax.imshow(Image.open(DATA_DIR / path))
    ax.set_title(c, fontsize=10)
plt.tight_layout()
plt.show()

### Поділ train / val / test (70 / 15 / 15)

За умовами лабораторної **всі експерименти мають використовувати однаковий поділ**.
Тому поділ зберігаємо у `split.csv` і при повторному запуску просто читаємо його з диска.

In [ ]:
split_file = CKPT_DIR / 'split.csv'

if split_file.exists():
    df = pd.read_csv(split_file)
    print('Поділ завантажено з', split_file)
else:
    train_df, tmp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
    val_df, test_df = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df['label'], random_state=SEED)
    df = pd.concat([train_df.assign(split='train'), val_df.assign(split='val'), test_df.assign(split='test')])
    df = df.reset_index(drop=True)
    df.to_csv(split_file, index=False)
    print('Поділ збережено в', split_file)

idx_train = np.where(df.split == 'train')[0]
idx_val = np.where(df.split == 'val')[0]
idx_test = np.where(df.split == 'test')[0]
print(f'train: {len(idx_train)}, val: {len(idx_val)}, test: {len(idx_test)}')

# Перевірка стратифікації: частка кожного класу в кожній вибірці майже однакова
pd.crosstab(df['class'], df['split'], normalize='columns').round(3).head(10)

### Кешування зображень у пам'яті

Оригінальні зображення мають розмір 600×600, і декодування JPEG на кожній епосі на 2 ядрах Colab — вузьке місце.
Тому **один раз** зменшуємо всі зображення до 224×224 і тримаємо їх у RAM як `uint8`-тензор (~1.5 ГБ).
Нормалізацію робимо вже на GPU. Аугментації (завдання 4) можна буде застосовувати до цих тензорів «на льоту».

In [ ]:
class PathDataset(Dataset):
    # Читає зображення з диска і зменшує до IMG_SIZE (використовується лише для побудови кешу)
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(DATA_DIR / self.paths[i]).convert('RGB')
        img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1)   # HWC -> CHW, uint8


IMAGES = torch.empty((len(df), 3, IMG_SIZE, IMG_SIZE), dtype=torch.uint8)
LABELS = torch.tensor(df['label'].values, dtype=torch.long)

loader = DataLoader(PathDataset(df['path']), batch_size=128, num_workers=NUM_WORKERS,
                    multiprocessing_context=MP_CONTEXT)
pos = 0
for batch in tqdm(loader, desc='Кешування зображень'):
    IMAGES[pos:pos + len(batch)] = batch
    pos += len(batch)
print(IMAGES.shape, f'{IMAGES.numel() / 1024**3:.2f} GB')

In [ ]:
class CachedDataset(Dataset):
    # Датасет поверх кешу IMAGES. transform (аугментація) застосовується лише до train-вибірки.
    def __init__(self, indices, transform=None):
        self.indices = np.asarray(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = self.indices[i]
        x = IMAGES[j]
        if self.transform is not None:
            x = self.transform(x)
        return x, LABELS[j]


def make_loader(indices, transform=None, shuffle=False):
    # Генератор з фіксованим seed: у парних експериментах (з/без аугментації, з/без дистиляції)
    # батчі йдуть в однаковому порядку -> чесне порівняння
    g = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(CachedDataset(indices, transform), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True, generator=g,
                      multiprocessing_context=MP_CONTEXT)


train_loader = make_loader(idx_train, shuffle=True)
val_loader = make_loader(idx_val)
test_loader = make_loader(idx_test)

# Нормалізація під ImageNet (такі статистики очікує попередньо навчена мережа). Робимо її на GPU.
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def prepare_batch(x, y):
    x = x.to(device, non_blocking=True).float().div_(255)
    return (x - MEAN) / STD, y.to(device, non_blocking=True)

### Допоміжні функції для навчання та оцінювання

* `fit` — навчання з вибором **найкращої епохи за validation accuracy** (ваги цієї епохи зберігаються).
  Параметр `loss_fn(logits, x, y)` дозволяє пізніше підставити дистиляційну функцію втрат (завдання 7).
* `evaluate_model` — рахує accuracy та macro-F1 на val і test, кількість параметрів і зберігає все у `results.json`.
  Також зберігає логіти на test — вони знадобляться для аналізу помилок (завдання 8, 9).

> Моделі та гіперпараметри обираємо **за validation**, а test використовуємо лише для фінального звіту.

In [ ]:
def ce_loss(logits, x, y):
    return F.cross_entropy(logits, y)


def keep_frozen_bn_in_eval(model):
    # Якщо шар BatchNorm заморожений, його running mean/var теж не повинні змінюватися,
    # тому переводимо такі шари в eval-режим навіть під час навчання.
    for m in model.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm) and not m.weight.requires_grad:
            m.eval()


def run_epoch(model, loader, loss_fn, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    if train:
        keep_frozen_bn_in_eval(model)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = prepare_batch(x, y)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
                loss = loss_fn(logits, x, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, optimizer, epochs, scheduler=None, loss_fn=ce_loss, name='model'):
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    history = defaultdict(list)
    best_acc, best_state, best_epoch = -1.0, None, 0
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, loss_fn, optimizer, scaler)
        va_loss, va_acc = run_epoch(model, val_loader, ce_loss)   # на валідації — звичайна CE
        if scheduler is not None:
            scheduler.step()
        for k, v in zip(['train_loss', 'train_acc', 'val_loss', 'val_acc'], [tr_loss, tr_acc, va_loss, va_acc]):
            history[k].append(v)
        if va_acc > best_acc:
            best_acc, best_epoch = va_acc, epoch
            best_state = copy.deepcopy(model.state_dict())
        print(f'[{name}] epoch {epoch:2d}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | '
              f'val loss {va_loss:.4f} acc {va_acc:.4f} | {time.time() - t0:.0f}s')
    print(f'Найкраща епоха: {best_epoch}, val acc = {best_acc:.4f}')
    model.load_state_dict(best_state)
    torch.save(best_state, CKPT_DIR / f'{name}.pt')
    with open(CKPT_DIR / f'{name}_history.json', 'w') as f:
        json.dump(history, f)
    return dict(history)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_logits, all_y = [], []
    for x, y in loader:
        x, y = prepare_batch(x, y)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
        all_logits.append(logits.float().cpu())
        all_y.append(y.cpu())
    return torch.cat(all_logits), torch.cat(all_y)


def count_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


results_file = CKPT_DIR / 'results.json'
results = json.load(open(results_file)) if results_file.exists() else {}


def evaluate_model(model, name):
    row = {}
    for split, loader in [('val', val_loader), ('test', test_loader)]:
        logits, y = predict(model, loader)
        pred = logits.argmax(1)
        row[f'{split}_acc'] = accuracy_score(y, pred)
        row[f'{split}_macro_f1'] = f1_score(y, pred, average='macro')
        if split == 'test':
            torch.save({'logits': logits, 'y': y}, CKPT_DIR / f'{name}_test_logits.pt')
    row['params_M'] = count_params(model) / 1e6
    results[name] = row
    with open(results_file, 'w') as f:
        json.dump(results, f, indent=2)
    return pd.Series(row, name=name)


def plot_history(history, title='', split_at=None):
    # split_at — номер епохи, після якої змінився режим навчання (для fine-tuning)
    epochs = np.arange(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, metric in zip(axes, ['loss', 'acc']):
        ax.plot(epochs, history[f'train_{metric}'], 'o-', label='train')
        ax.plot(epochs, history[f'val_{metric}'], 'o-', label='val')
        if split_at is not None:
            ax.axvline(split_at + 0.5, color='gray', ls='--', label='розморожування layer4')
        ax.set_xlabel('епоха')
        ax.set_title(f'{title}: {metric}')
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Попередньо навчена ResNet18 із замороженою основою

Беремо **ResNet18**, навчену на ImageNet (11.7 млн параметрів). Вона достатньо легка для Colab
і має чітку структуру блоків (`layer1 … layer4`), що зручно для fine-tuning у завданні 5.

* Замінюємо останній шар `fc` (1000 класів ImageNet) на `Linear(512 → 30)`.
* Заморожуємо всі ваги, крім нового `fc` — мережа працює як фіксований екстрактор ознак.
* Початкові ваги моделі зберігаємо в `resnet18_init.pt`: в експерименті з аугментацією (завдання 4)
  потрібно стартувати з **тих самих** ваг нового класифікатора.

In [ ]:
def build_resnet18(num_classes=NUM_CLASSES):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)   # новий класифікаційний шар
    return model


def freeze_backbone(model, trainable=('fc',)):
    # Заморожує всі параметри, крім модулів, назви яких починаються з trainable
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith(trainable)


set_seed()
model_fe = build_resnet18()
torch.save(model_fe.state_dict(), CKPT_DIR / 'resnet18_init.pt')
freeze_backbone(model_fe, trainable=('fc',))
model_fe = model_fe.to(device)

print(f'Усього параметрів:   {count_params(model_fe):,}')
print(f'Навчаються:          {count_params(model_fe, trainable_only=True):,}')

In [ ]:
# Гіперпараметри експерименту 3. ТІ САМІ значення треба використати в завданні 4 (бюджет навчання однаковий).
EPOCHS_FE = 10
LR_FE = 1e-3

optimizer = torch.optim.Adam([p for p in model_fe.parameters() if p.requires_grad], lr=LR_FE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FE)

hist_fe = fit(model_fe, train_loader, val_loader, optimizer, EPOCHS_FE, scheduler, name='exp3_frozen')

In [ ]:
plot_history(hist_fe, 'Exp 3: ResNet18, заморожена основа')
evaluate_model(model_fe, 'exp3_frozen')

**Висновок (заповнити після запуску):** _яку точність дає лише новий класифікатор поверх ознак ImageNet;
чи є розрив між train і val (ознаки перенавчання); чи вийшла крива val на плато._

## 4. Навчання з аугментацією — 📝 заготовка

### Поради щодо виконання

**Яку аугментацію обрати і чому.** AID — це **аерознімки, зроблені згори**, тому:

| Перетворення | Чому підходить |
|---|---|
| `RandomHorizontalFlip`, `RandomVerticalFlip` | у знімка згори немає «верху» і «низу» — віддзеркалена сцена лишається тим самим класом |
| повороти на 0/90/180/270° | те саме: орієнтація сцени довільна; повороти на кратні 90° не створюють чорних кутів |
| `RandomResizedCrop(224, scale=(0.7, 1.0))` | імітує різну висоту зйомки / масштаб; **не** брати малий `scale` — класи на кшталт `center`, `square`, `stadium`, `dense/medium/sparse residential` розрізняються саме контекстом і щільністю забудови |
| `ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1)` | різне освітлення, сезон, сенсор; **hue** не чіпати або дуже слабо — колір важливий (`bareLand`/`desert`, `farmland`/`meadow`, `beach`/`pond`) |

Можна порівняти з `v2.RandAugment` / `v2.TrivialAugmentWide`, але для аерознімків ручний набір вище зазвичай кращий.

**Умови чесного порівняння з exp3:** той самий поділ (`idx_train/val/test`), ті самі початкові ваги (`resnet18_init.pt`),
ті самі `EPOCHS_FE`, `LR_FE`, оптимізатор і scheduler. Аугментація — **лише** для train (`val_loader`/`test_loader` не змінюються).

**Що очікувати й про що писати у висновку.** При замороженій основі ефект аугментації зазвичай невеликий:
train accuracy стане нижчою (задача важча), а розрив train–val зменшиться. Ознаки перенавчання: train acc росте,
а val loss перестає падати / зростає. Варто порівняти криві exp3 і exp4 на одному графіку.
Якщо exp4 виявиться кращим за val — логічно робити fine-tuning (завдання 5) саме від нього і з аугментацією.

In [ ]:
# ===== Завдання 4: заготовка =====

# train_aug = v2.Compose([
#     v2.RandomHorizontalFlip(),
#     v2.RandomVerticalFlip(),
#     v2.RandomChoice([v2.RandomRotation((a, a)) for a in (0, 90, 180, 270)]),
#     v2.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), antialias=True),
#     v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
# ])
# train_loader_aug = make_loader(idx_train, transform=train_aug, shuffle=True)

# TODO: показати приклади аугментацій — одне й те саме зображення IMAGES[idx_train[0]],
#       пропущене через train_aug 8 разів (imshow(x.permute(1, 2, 0)))

# TODO: та сама модель і ті самі початкові ваги, що й у exp3
# set_seed()
# model_aug = build_resnet18()
# model_aug.load_state_dict(torch.load(CKPT_DIR / 'resnet18_init.pt'))
# freeze_backbone(model_aug, trainable=('fc',))
# model_aug = model_aug.to(device)
# optimizer = ...   # як у exp3: Adam, LR_FE
# scheduler = ...   # як у exp3: CosineAnnealingLR, T_max=EPOCHS_FE
# hist_aug = fit(model_aug, train_loader_aug, val_loader, optimizer, EPOCHS_FE, scheduler, name='exp4_frozen_aug')
# plot_history(hist_aug, 'Exp 4: ResNet18 + аугментація')
# evaluate_model(model_aug, 'exp4_frozen_aug')

# TODO: таблиця pd.DataFrame(results).T для exp3 vs exp4 + висновок про вплив аугментації

## 5. Fine-tuning: розморожуємо останній згортковий блок

Беремо найкращі ваги з exp3 і розморожуємо **`layer4`** (останній згортковий блок ResNet18, ~8.4 млн параметрів) разом із `fc`.
Навчання продовжуємо з **меншим learning rate** (`1e-4` замість `1e-3`): попередньо навчені ваги вже хороші,
і великий крок їх «зламав» би (catastrophic forgetting). BatchNorm-шари в `layer4` тепер теж оновлюють свої статистики
під наш датасет.

> Коли буде виконане завдання 4: якщо exp4 кращий за validation, тут варто стартувати з `exp4_frozen_aug.pt`
> і навчати з `train_loader_aug`.

In [ ]:
set_seed()
model_ft = build_resnet18()
model_ft.load_state_dict(torch.load(CKPT_DIR / 'exp3_frozen.pt'))   # стартуємо з найкращої моделі exp3
freeze_backbone(model_ft, trainable=('layer4', 'fc'))
model_ft = model_ft.to(device)

print(f'Навчаються: {count_params(model_ft, trainable_only=True):,} з {count_params(model_ft):,} параметрів')

EPOCHS_FT = 10
LR_FT = 1e-4   # у 10 разів менший, ніж у exp3

optimizer = torch.optim.Adam([p for p in model_ft.parameters() if p.requires_grad], lr=LR_FT, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FT)

hist_ft = fit(model_ft, train_loader, val_loader, optimizer, EPOCHS_FT, scheduler, name='exp5_finetune')

In [ ]:
# Криві навчання «до» і «після»: перші EPOCHS_FE епох — exp3, далі — fine-tuning
hist_both = {k: hist_fe[k] + hist_ft[k] for k in hist_fe}
plot_history(hist_both, 'Exp 3 → Exp 5', split_at=EPOCHS_FE)

evaluate_model(model_ft, 'exp5_finetune')
pd.DataFrame(results).T.loc[['exp3_frozen', 'exp5_finetune']].round(4)

**Висновок (заповнити після запуску):** _на скільки зросли accuracy / macro-F1 після розморожування `layer4`;
чи збільшився розрив між train і val (тепер навчається ~8.4 млн параметрів замість ~15 тис., тож ризик перенавчання вищий);
на якій епосі досягнуто найкращого val-результату._

## 6. Компактна CNN — модель-учень (звичайне навчання) — 📝 заготовка

### Поради щодо виконання

**Архітектура учня.** Рекомендую невелику CNN, яка навчається **з нуля** (без ImageNet-ваг) — тоді ефект дистиляції
в завданні 7 буде помітнішим і «чистим»:

```
вхід 3×224×224
[Conv3×3 → BN → ReLU] ×2, 32 канали  → MaxPool   -> 112×112
[Conv3×3 → BN → ReLU] ×2, 64 канали  → MaxPool   -> 56×56
[Conv3×3 → BN → ReLU] ×2, 128 каналів → MaxPool  -> 28×28
[Conv3×3 → BN → ReLU] ×2, 256 каналів → MaxPool  -> 14×14
GlobalAvgPool → Dropout(0.3) → Linear(256 → 30)
```
≈ **1.2 млн параметрів** — приблизно в 10 разів менше за ResNet18 (11.2 млн).
Перший шар можна зробити зі `stride=2`, щоб пришвидшити навчання на 224×224.
Альтернатива — `models.mobilenet_v3_small(weights=None, num_classes=30)` (~1 млн параметрів).
Якщо брати MobileNetV3 з ImageNet-вагами — це вже теж transfer learning, і це треба вказати у звіті.

**Навчання.** З нуля мережі потрібно більше епох (30–40), AdamW (`lr=1e-3`, `weight_decay=5e-4`) + cosine/OneCycle,
та сама аугментація, що й у кращого вчителя (з нуля без аугментації учень швидко перенавчиться на ~7000 зображень).
Обов'язково зберегти початкові ваги учня (`student_init.pt`) — учень із дистиляцією має стартувати з **тих самих** ваг.

In [ ]:
# ===== Завдання 6: заготовка =====

# class SmallCNN(nn.Module):
#     def __init__(self, num_classes=NUM_CLASSES):
#         super().__init__()
#         ...   # TODO: блоки Conv-BN-ReLU (див. схему вище)
#     def forward(self, x):
#         ...

# set_seed()
# student = SmallCNN()
# torch.save(student.state_dict(), CKPT_DIR / 'student_init.pt')   # ті самі ваги знадобляться в завданні 7
# student = student.to(device)
# print(f'Параметрів учня: {count_params(student):,}')

# EPOCHS_ST = 30
# optimizer = torch.optim.AdamW(student.parameters(), lr=1e-3, weight_decay=5e-4)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_ST)
# hist_st = fit(student, train_loader_aug, val_loader, optimizer, EPOCHS_ST, scheduler, name='exp6_student')
# plot_history(hist_st, 'Exp 6: учень (звичайне навчання)')
# evaluate_model(student, 'exp6_student')

## 7. Учень із дистиляцією знань — 📝 заготовка

### Поради щодо виконання

**Учитель** — модель з **найвищою val accuracy** серед exp3, exp4, exp5 (вибір за validation, не за test!).
Імовірно, це буде `exp5_finetune`.

**Функція втрат** (Hinton et al., 2015):

$$L = \alpha \cdot T^2 \cdot \mathrm{KL}\big(\mathrm{softmax}(z_t/T)\,\|\,\mathrm{softmax}(z_s/T)\big) + (1-\alpha)\cdot \mathrm{CE}(z_s, y)$$

* **T (температура)** «розм'якшує» розподіл учителя, щоб учень бачив, які класи вчитель вважає схожими
  (наприклад, `dense` ↔ `medium residential`). Стартове значення **T = 4**.
* **α (вага дистиляційної складової)** — стартове значення **α = 0.7**.
* Множник `T²` компенсує зменшення градієнтів через ділення логітів на T.
* За бажанням підібрати T ∈ {2, 4, 8} і α ∈ {0.5, 0.7, 0.9} **за validation** і вказати обрані значення у звіті.

**Технічні моменти:**
* учитель — у `eval()`, без градієнтів; на вхід отримує **той самий аугментований батч**, що й учень;
* функція втрат має сигнатуру `loss_fn(logits, x, y)`, тому її можна прямо передати у `fit(..., loss_fn=kd_loss)`;
* чесне порівняння з exp6: ті самі `student_init.pt`, епохи, оптимізатор, scheduler, аугментація і порядок батчів
  (`make_loader` з тим самим seed). Відрізняється **лише** функція втрат.

**Як порівнювати моделі.** Для вчителя і двох учнів `evaluate_model` вже рахує на одному й тому самому test:
accuracy, macro-F1 і кількість параметрів. Додатково варто вказати розмір ваг (МБ) і час інференсу на одне зображення.
Висновок: на скільки учень із KD наблизився до вчителя порівняно зі звичайним учнем і у скільки разів він менший.

In [ ]:
# ===== Завдання 7: заготовка =====

# Вибір учителя за validation accuracy
# teacher_name = max(['exp3_frozen', 'exp4_frozen_aug', 'exp5_finetune'],
#                    key=lambda k: results.get(k, {}).get('val_acc', -1))
# teacher = build_resnet18()
# teacher.load_state_dict(torch.load(CKPT_DIR / f'{teacher_name}.pt'))
# teacher = teacher.to(device).eval()

# T, ALPHA = 4.0, 0.7

# def kd_loss(student_logits, x, y):
#     with torch.no_grad():
#         teacher_logits = teacher(x)
#     soft = F.kl_div(F.log_softmax(student_logits / T, dim=1),
#                     F.softmax(teacher_logits / T, dim=1), reduction='batchmean') * T * T
#     hard = F.cross_entropy(student_logits, y)
#     return ALPHA * soft + (1 - ALPHA) * hard

# set_seed()
# student_kd = SmallCNN()
# student_kd.load_state_dict(torch.load(CKPT_DIR / 'student_init.pt'))   # ті самі початкові ваги, що в exp6
# ... той самий оптимізатор / scheduler / EPOCHS_ST
# hist_kd = fit(student_kd, train_loader_aug, val_loader, optimizer, EPOCHS_ST, scheduler,
#               loss_fn=kd_loss, name='exp7_student_kd')
# evaluate_model(student_kd, 'exp7_student_kd')

# Порівняння: pd.DataFrame(results).T.loc[[teacher_name, 'exp6_student', 'exp7_student_kd']]

## 8. Аналіз помилок і Grad-CAM — 📝 заготовка

**Поради:**
* Логіти на test для кожної моделі вже збережені в `CKPT_DIR/<name>_test_logits.pt` (порядок = `idx_test`).
* Матриці помилок — `sklearn.metrics.ConfusionMatrixDisplay` з `normalize='true'` (30 класів → `figsize` ~ 14×14).
  Шукати пари класів, які плутаються найчастіше (очікувано: `dense/medium/sparseResidential`, `school`/`commercial`, `bareLand`/`desert`).
* Приклади помилок: зображення з `IMAGES[idx_test[i]]` з підписом «правильний → передбачений (впевненість)».
* Grad-CAM: найпростіше через бібліотеку `pip install grad-cam` (`from pytorch_grad_cam import GradCAM`).
  Цільовий шар: для ResNet18 — `model.layer4[-1]`, для учня — останній згортковий блок.
  Показати 3–4 правильні та 3–4 неправильні прогнози і пояснити, на які області дивиться модель
  (на об'єкт класу чи на фон / сусідні об'єкти).

In [ ]:
# ===== Завдання 8: заготовка =====
# TODO: матриці помилок для вчителя та двох учнів
# TODO: приклади неправильних прогнозів
# TODO: Grad-CAM для кількох правильних і неправильних прогнозів

## 9. Які класи покращились / погіршились (exp 4 та exp 7) — 📝 заготовка

**Поради:**
* Пари для порівняння: exp3 → exp4 (вплив аугментації) і exp6 → exp7 (вплив дистиляції).
* Per-class F1 або recall — `classification_report(y, pred, output_dict=True)`; побудувати barplot різниці по класах.
* **Виправлені помилки** — `(pred_before != y) & (pred_after == y)`, **нові помилки** — `(pred_before == y) & (pred_after != y)`.
  Показати по кілька прикладів кожного типу.

In [ ]:
# ===== Завдання 9: заготовка =====

## 10. Підсумкова таблиця і висновки — 📝 заготовка

**Поради:**
* Таблиця: `pd.DataFrame(results).T` — val/test accuracy, macro-F1, кількість параметрів (+ розмір у МБ, час інференсу).
* **Найточніша** — за test accuracy / macro-F1; **найкомпактніша** — за кількістю параметрів.
* **Найстійкіша** — оцінити всі моделі на test зі спотвореннями (гаусів шум, розмиття `v2.GaussianBlur`,
  зміна яскравості/контрасту, JPEG-стиснення) і порівняти **падіння accuracy** відносно чистого test.
  Також можна врахувати розрив train–val (стійкість до перенавчання).

In [ ]:
# ===== Завдання 10: заготовка =====
# pd.DataFrame(results).T.round(4)